# Сравнение стратегий — финальный отчёт

Что сравниваем (полный OOS 2010-01 → 2017-11, 94 monthly rebalances, top/bot 10% L/S, 5bp комиссия):

| блок | стратегия | модель | таргет |
|---|---|---|---|
| baseline | `ridge` | Ridge regression (L2) | `target_reg` |
| baseline | `logreg` | Logistic regression + L2 | `target_clf` |
| DL-reg   | `mlp` (×5 seeds) | 2-layer MLP, Smooth-L1, AdamW | `target_reg` |
| DL-reg   | `lstm` (×5 seeds) | 1-layer LSTM, T=12 monthly snapshots | `target_reg` |
| DL-clf   | `mc_dropout` (×5 seeds) | MC-Dropout MLP, K=30 forward-passes | `target_clf` |
| DL-clf   | `mc_dropout_filtered` (×5 seeds) | + uncertainty filter (top 50% по σ) | `target_clf` |

**Метрики**: Sharpe, geometric mean, vol, max drawdown, alpha vs SPX (с p-value), IR, final NAV.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation.plots import (
    plot_comparison, plot_drawdown, plot_equity, plot_rolling_sharpe,
)

RESULTS = Path('../results')
assert RESULTS.exists(), 'Сначала запустите run_baselines/run_dl_reg/run_dl_clf.'
print('Найдены папки:', sorted(p.name for p in RESULTS.iterdir() if p.is_dir() and p.name != '_summary'))

## 1. Сводная таблица метрик

Берём по одной строке на стратегию: для baseline — собственная папка, для DL — агрегат `*_agg/agg_metrics.csv` с усреднением по сидам.

In [ ]:
STRATEGIES = {
    'ridge': {'kind': 'single', 'dir': 'ridge'},
    'logreg': {'kind': 'single', 'dir': 'logreg'},
    'mlp': {'kind': 'agg', 'dir': 'mlp_agg'},
    'lstm': {'kind': 'agg', 'dir': 'lstm_agg'},
    'mc_dropout': {'kind': 'agg', 'dir': 'mc_dropout_agg'},
    'mc_dropout_filtered': {'kind': 'agg', 'dir': 'mc_dropout_filtered_agg'},
}

rows = []
for name, spec in STRATEGIES.items():
    d = RESULTS / spec['dir']
    if not d.exists():
        rows.append({'strategy': name, 'present': False})
        continue
    if spec['kind'] == 'single':
        m = pd.read_csv(d / 'metrics.csv', index_col=0)['value']
    else:
        m = pd.read_csv(d / 'agg_metrics.csv', index_col=0)['mean']
    row = {'strategy': name, 'present': True}
    for k in ['sharpe', 'geom_avg_total_r', 'std_xs_r', 'max_dd', 'alpha_benchmark', 'alpha_benchmark_pvalue', 'ir_benchmark', 'final_nav']:
        try:
            row[k] = float(m.get(k))
        except Exception:
            row[k] = np.nan
    rows.append(row)

summary = pd.DataFrame(rows).set_index('strategy')
summary = summary[summary['present']].drop(columns=['present'])
summary = summary.sort_values('sharpe', ascending=False)
summary.style.format({
    'sharpe': '{:.3f}', 'geom_avg_total_r': '{:.2%}', 'std_xs_r': '{:.2%}',
    'max_dd': '{:.2%}', 'alpha_benchmark': '{:.2%}', 'alpha_benchmark_pvalue': '{:.3f}',
    'ir_benchmark': '{:.3f}', 'final_nav': '{:.3f}',
})

## 2. Equity-кривые: все стратегии vs SPX

In [ ]:
curves = {}
benchmark = None
for name, spec in STRATEGIES.items():
    d = RESULTS / spec['dir']
    if not d.exists():
        continue
    fname = 'mean_returns.parquet' if spec['kind'] == 'agg' else 'total_returns.parquet'
    curves[name] = pd.read_parquet(d / fname).iloc[:, 0]
    if benchmark is None:
        bench_p = d / 'benchmark_returns.parquet'
        if not bench_p.exists():
            for s in STRATEGIES.values():
                p = RESULTS / s['dir'].replace('_agg', '_s0') / 'benchmark_returns.parquet'
                if p.exists():
                    bench_p = p
                    break
        if bench_p.exists():
            benchmark = pd.read_parquet(bench_p).iloc[:, 0]

fig = plot_comparison(curves, benchmark, benchmark_name='SPX (total)', title='All strategies vs SPX')
plt.show()

## 3. Drawdowns по стратегиям

Один график на стратегию, чтобы хорошо различались просадки.

In [ ]:
n = len(curves)
fig, axes = plt.subplots(n, 1, figsize=(11, 2.4 * n), sharex=True)
if n == 1:
    axes = [axes]
for ax, (name, ret) in zip(axes, curves.items()):
    eq = (1 + ret.fillna(0)).cumprod()
    dd = eq / eq.cummax() - 1
    ax.fill_between(dd.index, dd.values, 0, color='#c0392b', alpha=0.4)
    ax.plot(dd.index, dd.values, color='#c0392b', lw=0.8)
    ax.set_title(f'{name}: max DD = {dd.min():.1%}')
    ax.grid(alpha=0.3, lw=0.4)
fig.tight_layout()
plt.show()

## 4. Rolling Sharpe (252 дней)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, ret in curves.items():
    rs = ret.rolling(252).mean() / ret.rolling(252).std() * np.sqrt(252)
    ax.plot(rs.index, rs.values, label=name, lw=1.1)
ax.axhline(0, color='black', lw=0.5)
ax.grid(alpha=0.3, lw=0.4)
ax.legend(frameon=False)
ax.set_title('Rolling 252d Sharpe (annualized)')
fig.tight_layout()
plt.show()

## 5. Robustness: распределение per-seed метрик для DL-стратегий

In [ ]:
per_seed = {}
for name, spec in STRATEGIES.items():
    if spec['kind'] != 'agg':
        continue
    d = RESULTS / spec['dir']
    if not d.exists():
        continue
    df = pd.read_csv(d / 'per_seed_metrics.csv').set_index('seed')
    per_seed[name] = df

if per_seed:
    metric_grid = ['sharpe', 'final_nav', 'alpha_benchmark', 'ir_benchmark']
    fig, axes = plt.subplots(1, len(metric_grid), figsize=(4.5 * len(metric_grid), 4))
    for ax, metric in zip(axes, metric_grid):
        data = [df[metric].values for df in per_seed.values()]
        ax.boxplot(data, tick_labels=list(per_seed.keys()))
        ax.set_title(metric)
        ax.grid(alpha=0.3, lw=0.4)
        ax.tick_params(axis='x', rotation=15)
    fig.tight_layout()
    plt.show()
else:
    print('Multi-seed runs not found yet.')

## 6. Главные выводы

Заполните после прогона полного OOS:

- Лучшая стратегия по **risk-adjusted alpha vs SPX** (IR_benchmark): ...
- Лучшая стратегия по **absolute return** (final NAV): ...
- Robustness: разброс Sharpe по сидам — для каких моделей он мал, для каких большой?
- MC-Dropout uncertainty filter: помогает ли (сравнение `mc_dropout` vs `mc_dropout_filtered`)?
- DL vs baselines: оправдывает ли сложность DL прирост в IR?